In [ ]:
import multiprocessing as mp
from datetime import date
import pandas as pd
from gymnasium.vector import SyncVectorEnv
import torch
from torch.utils.tensorboard import SummaryWriter

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.env_stock_trading.env_forex_price_trailing import ForexPriceTrailingEnv
from datetime import datetime, date
import pandas as pd
import itertools
# from enum import StrEnum, auto
import numpy as np
from torch import optim
import gymnasium as gym
from torch.utils.tensorboard import SummaryWriter
from matplotlib import pyplot as plt
# import ray
# from ray import train
from torch.utils.tensorboard import SummaryWriter
# from ray.train import Checkpoint
# from ray import tune
# from ray.air import session
# from ray.tune import CLIReporter
# from ray.tune.schedulers import ASHAScheduler
import torch
import torch.nn as nn
from collections import deque
import random
import os
import torch.nn.functional as F
import torch.nn as nns
from copy import deepcopy
# from models import LSTM_QNet, ReplayBuffer
from gymnasium.wrappers import TimeLimit    
from stable_baselines3.common.type_aliases import Schedule

ImportError: cannot import name 'StrEnum' from 'enum' (/usr/lib/python3.10/enum.py)

In [33]:
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.dqn.policies import MlpPolicy, LstmDQNPolicy
from stable_baselines3 import DQN
from stable_baselines3.her.her_replay_buffer import HerReplayBuffer
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

In [3]:
def load_and_preprocess():
    # 1) tickers
    majors = [
        "EURUSD=X",
        "USDJPY=X",
        "GBPUSD=X",
        "AUDUSD=X",
        "USDCAD=X",
        "USDCHF=X",
        "NZDUSD=X",
    ]
    crosses = [
        "EURGBP=X",
        "EURJPY=X",
        "GBPJPY=X",
        "AUDJPY=X",
        "CADJPY=X",
        "EURAUD=X",
        "EURCAD=X",
        "EURCHF=X",
        "GBPCHF=X",
        "AUDCAD=X",
        "NZDJPY=X",
        "NZDCAD=X",
    ]
    cny = [
        "USDCNY=X",
        "EURCNY=X",
        "JPYCNY=X",
        "GBPCNY=X",
        "AUDCNY=X",
        "CADCNY=X",
        "CHFCNY=X",
        "NZDCNY=X",
    ]
    all_tickers = majors + crosses + cny
    seen = set()
    forex_ticks = [t for t in all_tickers if not (t in seen or seen.add(t))]

    # 2) download
    start_train = date(2000, 1, 1)
    end_train = date(2017, 1, 1)
    eval_span = (date(2016, 7, 1), date(2017, 1, 1))  # last 6 months
    test_span = (date(2017, 1, 1), date(2018, 6, 1))

    yfd = YahooDownloader(
        start_date=str(start_train), end_date=str(end_train), ticker_list=forex_ticks
    )
    df_train = add_fx_features(yfd.fetch_data())

    yfd2 = YahooDownloader(
        start_date=str(test_span[0]),
        end_date=str(test_span[1]),
        ticker_list=forex_ticks,
    )
    df_test = add_fx_features(yfd2.fetch_data())

    sample_df = df_train[df_train.tic == forex_ticks[0]].reset_index(drop=True)
    val_start_idx = sample_df[
        pd.to_datetime(sample_df.date) == pd.Timestamp(eval_span[0])
    ].index[0]
    val_length = int((eval_span[1] - eval_span[0]).days)

    return df_train, df_test, forex_ticks, val_start_idx, val_length


def add_fx_features_for_tick(g):
    g["close_prev"] = g.close.shift(1)
    g["high_prev"] = g.high.shift(1)
    g["low_prev"] = g.low.shift(1)
    g["x1"] = (g.close - g.close_prev) / g.close_prev
    g["x2"] = (g.high - g.high_prev) / g.high_prev
    g["x3"] = (g.low - g.low_prev) / g.low_prev
    g["x4"] = (g.high - g.close) / g.close
    g["x5"] = (g.close - g.low) / g.close
    return g


def add_fx_features(df, tic_col="tic"):
    return (
        df.groupby(tic_col, group_keys=False)
        .apply(add_fx_features_for_tick)
        .drop(columns=["close_prev", "high_prev", "low_prev"])
        .fillna(0)
    )


In [4]:
df_train, df_test, forex_ticks, val_start_idx, val_length = load_and_preprocess()
df_train

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (93926, 8)


/tmp/ipykernel_134052/269295929.py:82: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%**************

Shape of DataFrame:  (9915, 8)


/tmp/ipykernel_134052/269295929.py:82: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)


Price,date,close,high,low,open,volume,tic,day,x1,x2,x3,x4,x5
0,2000-01-03,0.625400,0.629000,0.620300,0.623900,0,EURGBP=X,0,0.000000,0.000000,0.000000,0.005756,0.008155
1,2000-01-03,101.690002,103.330002,101.309998,102.070000,0,USDJPY=X,0,0.000000,0.000000,0.000000,0.016127,0.003737
2,2000-01-04,0.628300,0.631400,0.623600,0.625300,0,EURGBP=X,1,0.004637,0.003816,0.005320,0.004934,0.007481
3,2000-01-04,103.139999,103.320000,101.470001,101.639999,0,USDJPY=X,1,0.014259,-0.000097,0.001579,0.001745,0.016192
4,2000-01-05,0.628400,0.633000,0.626800,0.628100,0,EURGBP=X,2,0.000159,0.002534,0.005131,0.007320,0.002546
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93921,2016-12-30,0.697204,0.697496,0.694493,0.697204,0,NZDUSD=X,4,0.006484,0.000488,0.003820,0.000418,0.003889
93922,2016-12-30,1.347810,1.349390,1.340240,1.347760,0,USDCAD=X,4,-0.005497,-0.004632,-0.005963,0.001172,0.005617
93923,2016-12-30,1.017100,1.022820,1.014400,1.016920,0,USDCHF=X,4,-0.010757,-0.005271,-0.007825,0.005624,0.002655
93924,2016-12-30,6.945300,6.954600,6.934500,6.954600,0,USDCNY=X,4,-0.001840,-0.000546,-0.001454,0.001339,0.001555


In [5]:
ddqn_writer = SummaryWriter("runs/stable_baselines3/dqn")
train_env = ForexPriceTrailingEnv(
    df_train,
    tic_col="tic",
    window=16,
    margin=0.02,
    step_frac=0.1,
    fee=2e-4,
    alpha_trail=0.97,
    alpha_pnl=0.85,
    alpha_fee=1.0,
    episode_len=1000,
    pick_new_pair_every=1,
    writer=ddqn_writer,
)
train_env = TimeLimit(train_env, max_episode_steps=600)

In [6]:
# Make sure the env is properly configured
check_env(train_env, warn=True, skip_render_check=True)

In [7]:
default_gpu = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 0:
    torch.cuda.set_device(default_gpu)

print(f"Running on device: {device}")

Running on device: cuda


In [8]:
dqn_mlp_model = DQN(
    policy="MlpPolicy",
    # policy_kwargs={"net_arch": [128, 128, 64]}
    env=train_env,
    learning_rate=1e-4,
    buffer_size=200_000,
    learning_starts=100,
    batch_size=512,
    tau=1,
    gamma=0.995,
    train_freq=4,
    gradient_steps=2,
    target_update_interval=1000,
    exploration_fraction=0.2,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    max_grad_norm=10,
    stats_window_size=100,
    tensorboard_log="runs/stable_baselines3/dqn",
    verbose=True,
    device=device
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [31]:
policy_kwargs = dict(
    window=16,
    feature_dim=5,
    lstm_hidden=128,
    fc_hidden=64
)

dqn_lstm_model = DQN(
    policy="LstmDQNPolicy",
    policy_kwargs=policy_kwargs,
    env=train_env,
    learning_rate=2e-4,
    buffer_size=200_000,
    learning_starts=100,
    batch_size=512,
    tau=1,
    gamma=0.995,
    train_freq=4,
    gradient_steps=2,
    target_update_interval=1000,
    exploration_fraction=0.2,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    max_grad_norm=10,
    stats_window_size=100,
    tensorboard_log="runs/stable_baselines3/dqn_lstm",
    verbose=True,
    device=device
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
checkpoint_callback = CheckpointCallback(
  save_freq=1000,
  save_path="checkpoints_sb3/",
  name_prefix="ddqn_lstm_sb3",
  save_replay_buffer=True,
  save_vecnormalize=True,
)

dqn_lstm_model.learn(
    total_timesteps=1_000_000,
    log_interval=10,
    tb_log_name="DDQN_LSTM",
    callback=checkpoint_callback
)

Logging to runs/stable_baselines3/dqn_lstm/DDQN_LSTM_2
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 600       |
|    ep_rew_mean      | -4.13e+05 |
|    exploration_rate | 0.97      |
| time/               |           |
|    episodes         | 10        |
|    fps              | 119       |
|    time_elapsed     | 50        |
|    total_timesteps  | 6000      |
| train/              |           |
|    learning_rate    | 0.0001    |
|    loss             | 657       |
|    n_updates        | 2948      |
-----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 600      |
|    ep_rew_mean      | -2.4e+05 |
|    exploration_rate | 0.941    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 119      |
|    time_elapsed     | 100      |
|    total_timesteps  | 12000    |
| train/              |          |
|    learning_rate  

KeyboardInterrupt: 

In [ ]:
agent_prices = []
lowers = []
uppers = []
closes = []

obs, _ = train_env.reset()
for _ in range(100):
    action, _ = dqn_lstm_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = train_env.step(action)
    agent_prices.append(info["agent_price"])
    lowers.append(info["lower"])
    uppers.append(info["upper"])
    closes.append(info["close"])

plt.figure(figsize=(10, 6))
plt.plot(range(100), agent_prices, label="Agent Price")
plt.plot(range(100), lowers, label="Lower")
plt.plot(range(100), uppers, label="Upper")
plt.plot(range(100), closes, label="Close")
plt.xlabel("Step")
plt.ylabel("Price")
plt.legend()
plt.title("Agent Price, Lower, Upper, and Close over Steps")
plt.show()

In [40]:
def make_env(
    df, tic_col, window, margin, step_frac, fee,
    alpha_trail, alpha_pnl, alpha_fee,
    episode_len, pick_new_pair_every, rank, name
):
    def _init():
        w = SummaryWriter("runs/stable_baselines3/dqn_lstm_vec")
        env = ForexPriceTrailingEnv(
            df,
            tic_col=tic_col,
            window=window,
            margin=margin,
            step_frac=step_frac,
            fee=fee,
            alpha_trail=alpha_trail,
            alpha_pnl=alpha_pnl,
            alpha_fee=alpha_fee,
            episode_len=episode_len,
            pick_new_pair_every=pick_new_pair_every,
            writer=w,
            name=name
        )
        env = TimeLimit(env, max_episode_steps=600)
        return env
    return _init

# 2) number of parallel environments
n_envs = 8
window = 32
# shared objects / kwargs
common_kwargs = dict(
    df=df_train,
    tic_col="tic",
    window=window,
    margin=0.02,
    step_frac=0.1,
    fee=2e-4,
    alpha_trail=0.97,
    alpha_pnl=0.85,
    alpha_fee=1.0,
    episode_len=1000,
    pick_new_pair_every=1,
)

# build the VecEnv
env_fns = [
    make_env(rank=i, name=f"Env_{i}", **common_kwargs)
    for i in range(n_envs)
]
train_env = SubprocVecEnv(env_fns)
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=False, clip_obs=10.)

# 3) define your model exactly as before, just pass train_env
policy_kwargs = dict(
    window=window,
    feature_dim=5,
    lstm_hidden=256,
    fc_hidden=128,
)

dqn_lstm_model = DQN(
    policy="LstmDQNPolicy",             # or "LstmDQNPolicy"
    policy_kwargs=policy_kwargs,
    env=train_env,
    learning_rate=2e-4,
    buffer_size=1_000_000,
    learning_starts=100,
    batch_size=2048,
    tau=1,
    gamma=0.995,
    train_freq=4,
    gradient_steps=-1,
    target_update_interval=1000,
    exploration_fraction=0.2,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    max_grad_norm=10,
    stats_window_size=100,
    tensorboard_log="runs/stable_baselines3/dqn_lstm_vec",
    verbose=1,
    device=device,
)

# scale total_timesteps by n_envs if you want the same per-env budget:
total_episodes = 1000
steps_per_ep   = 600
# total steps per env = total_episodes * steps_per_ep
# total across all envs = that * n_envs
total_timesteps = total_episodes * steps_per_ep * n_envs

checkpoint_callback = CheckpointCallback(
    save_freq=10_000,  # note: this is in *env steps per worker*
    save_path="checkpoints_sb3/",
    name_prefix="ddqn_lstm_sb3_vec",
    save_replay_buffer=True,
    save_vecnormalize=True,
)

dqn_lstm_model.learn(
    total_timesteps=total_timesteps,
    log_interval=10,
    tb_log_name="DDQN_LSTM_VEC",
    callback=checkpoint_callback,
)


Using cuda device
Logging to runs/stable_baselines3/dqn_lstm_vec/DDQN_LSTM_VEC_1
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.99     |
| time/               |          |
|    episodes         | 10       |
|    fps              | 57       |
|    time_elapsed     | 168      |
|    total_timesteps  | 9600     |
| train/              |          |
|    learning_rate    | 0.0002   |
|    loss             | 139      |
|    n_updates        | 9472     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.985    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 56       |
|    time_elapsed     | 255      |
|    total_timesteps  | 14400    |
| train/              |          |
|    learning_rate    | 0.0002   |
|    loss             | 115      |
|    n_updates        | 14272    |
----------------------------------
---------

OSError: [Errno 28] No space left on device